# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
# IMPORTS
# Import libraries for data analysis and visualization (provided)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# LOAD THE DATA
# encoding='latin-1' --> handles special characters in the provided data file
# low_memory=False --> prevents mixed-datatype warnings when reading large files
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

# Preview the first 5 rows to understand the structure of the data:
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# DATA TYPES
# Display column names, non-null counts, and datatypes for all 31 columns
# (Note: Event.Date is stored as text/object and will need to be converted to datetime)
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [4]:
# MISSING VALUES
# Count missing values (NaN) in each column, sorted from most to least missing
# (Columns like Schedule, Air.carrier, and Aircraft.Category have large amounts of missing data)
df.isna().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

In [5]:
# SUMMARY STATISTICS for numeric columns (engines and injury counts)
# Medians are much lower than maximums --> most accidents involve small aircraft
# with few passengers, while a few large-aircraft accidents create extreme values

df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# INSPECT RELVANT COLUMNS that we need to filter on, including missing values (dropna=False)
# Aircraft.Category: 56,602 values are missing, so dropping them would lose most of the data
# Amateur.Built: "No" = professional build, "Yes" = homebuilt, NaN = unknown

print(df['Aircraft.Category'].value_counts(dropna=False))
print()
print(df['Amateur.Built'].value_counts(dropna=False))

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64


In [7]:
# DATE FILTER

# Convert Event.Date from text to datetime so dates can be compared
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

# Client assumes a max 40-year aircraft lifetime, data ends in 2023
# so keep only events from 1983 onward. .copy() avoids SettingWithCopyWarning later.
df = df[df['Event.Date'] >= '1983-01-01'].copy()

# Check remaining rows: 85,289 (3,600 pre-1983 events removed)
df.shape

(85289, 31)

In [8]:
# PROFESSIONAL BUILDS

# Client only wants professionally-built aircraft
# Keeping rows with Amateur.Built is 'No' (also dropping rows with unknown build)
df = df[df['Amateur.Built'] == 'No'].copy()

# Check number of rows remaining (76960)
df.shape

(76960, 31)

In [9]:
# CLEANING UP THE MAKE NAMES

# Standardize names so "CESSNA" and "Cessna" count as one make
df ['Make'] = df['Make'].str.strip().str.title()

# View the 10 most common to confirm clean-up worked:
df['Make'].value_counts().head(10)

Make
Cessna      25740
Piper       14099
Beech        5094
Boeing       2649
Bell         2594
Mooney       1271
Robinson     1197
Grumman      1064
Bellanca      978
Hughes        877
Name: count, dtype: int64

In [10]:
# IMPUTING AIRCRAFT CATEGORY & FILTER TO AIRPLANES-ONLY

# Find each Make's most common (known) category
make_category = (
    df.dropna(subset=['Aircraft.Category'])
    .groupby('Make')['Aircraft.Category']
    .agg(lambda x: x.mode()[0])
)

# Fill missing categories using the Make's most-common category
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(df['Make'].map(make_category))

# Keep only airplanes, as requested
df = df[df['Aircraft.Category'] == 'Airplane'].copy()
df.shape

(66326, 31)

**Filtering summary:**
- Kept events from 1983 onward (40-year max aircraft lifetime): 85,289 rows
- Kept professional builds only (`Amateur.Built == 'No'`): 76,960 rows
- Standardized `Make` names to fix inconsistent capitalization
- `Aircraft.Category` was missing for most rows, so missing values were filled with each Make's most common category. Kept airplanes only: **66,326 rows**

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [11]:
# INSPECTING INJURY COLUMNS

# The four columns that together estimate everyone on board
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Count missing values in each column:
df[injury_cols].isna().sum()

Total.Fatal.Injuries      8539
Total.Serious.Injuries    9458
Total.Minor.Injuries      8979
Total.Uninjured           4022
dtype: int64

In [12]:
# BUILDING THE FATAL/SERIOUS INJURY METRIC:

# Assume blank injury counts means none reported (0)
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total people on board:
df['Total.Passengers'] = df[injury_cols].sum(axis=1)    # adds across each row

# Drop rows with 0 people on board (since the rate would be x/0 -> can't calc)
df = df[df['Total.Passengers'] > 0].copy()

# Fraction of people fatally/seriously injured:
df['Fatal.Serious.Metric'] = (
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'])
    / df['Total.Passengers']
)

df['Fatal.Serious.Metric'].describe()


count    65200.000000
mean         0.269344
std          0.428135
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: Fatal.Serious.Metric, dtype: float64

**Injury metric:**
- Missing injury counts were filled with 0 (assumed none reported).
- `Total.Passengers` = fatal + serious + minor + uninjured (estimate of everyone on board).
- Dropped 1,126 rows with 0 total passengers (no usable injury data): 65,200 rows remain.
- `Fatal.Serious.Metric` = (fatal + serious) / total passengers: the fraction of people on board fatally or seriously injured (0 to 1).

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [13]:
# INSPECTING 'AIRCRAFT DAMAGE'

# View each damage category and how often it appears (includes blanks)
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    47852
Destroyed      13255
NaN             2016
Minor           2003
Unknown           74
Name: count, dtype: int64

In [14]:
# CLEANING 'AIRCRAFT DAMAGE' & FLAGGING DESTROYED AIRCRAFT

# Drop blank and unknown damage (since not sure if it's destroyed)
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()

# Flagging: 1 = destroyed, 0 = not destroyed
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

df['Destroyed'].value_counts()

Destroyed
0    49855
1    13255
Name: count, dtype: int64

**Aircraft damage:**
- Dropped 2,090 rows where damage was 'NaN' or 'Unknown' (can't determine if destroyed).
- `Destroyed` = 1 if the aircraft was destroyed, 0 otherwise (Substantial or Minor). The mean of this column gives the destruction rate for any group.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [15]:
# INSPECTING THE MAKE COLUMN

# Missing and unique Makes:
print('Missing:', df['Make'].isna().sum())
print('Unique Make:', df['Make'].nunique())     # counts how many different values a column has

# Looking for name variations of major manufacturers:
major = 'Cessna|Piper|Beech|Boeing|Airbus|Mooney|Cirrus|Embraer|Bombardier|Mcdonnell'
df[df['Make'].str.contains(major, na=False)]['Make'].value_counts().head(30)

Missing: 1
Unique Make: 1050


Make
Cessna                           25388
Piper                            13936
Beech                             5001
Boeing                            1300
Mooney                            1256
Mcdonnell Douglas                  394
Cirrus Design Corp                 219
Embraer                            144
Cirrus                             140
Airbus Industrie                    73
Cirrus Design Corp.                 72
Airbus                              65
Boeing Stearman                     50
Bombardier                          45
Mooney Aircraft Corp.               34
Bombardier Inc                      33
Piper Aircraft Inc                  25
Cessna Aircraft Co                  24
Beechcraft                          23
Bombardier, Inc.                    15
Hawker Beechcraft                   14
Cirrus Design Corporation           13
Hawker Beechcraft Corp              13
Cessna Aircraft Company             11
Cessna Aircraft                     10
Mooney Airplane Co I

In [16]:
# CLEANING THE MAKE NAMES:

# Drop the missing Make
df = df.dropna(subset=['Make']).copy()

# Remove company suffixes and punctuation:
suffixes = r'[.,]|\b(Aircraft|Corporation|Corp|Company|Co|Inc|Industrie|Design|Llc|Ltd)\b'
df['Make'] = (df['Make'].str.replace(suffixes, '', regex=True)
              .str.replace(r'\s+', ' ', regex=True)
              .str.strip())

# Merge Beech's name variations:
df['Make'] = df['Make'].replace({
    'Beechcraft': 'Beech', 'Hawker Beechcraft': 'Beech',
    'Hawker-Beechcraft': 'Beech', 'Hawker Beech': 'Beech',
    'Mooney Airplane': 'Mooney',
    'New Piper': 'Piper',
    'Embraer-Empresa Brasileira De': 'Embraer', 'Embraer S A': 'Embraer'
})

# Check: unique makes & same manufacturer list as before
print('Unique Make:', df['Make'].nunique())
df[df['Make'].str.contains(major, na=False)]['Make'].value_counts().head(20)


Unique Make: 886


Make
Cessna                  25434
Piper                   13989
Beech                    5081
Mooney                   1303
Boeing                   1302
Cirrus                    449
Mcdonnell Douglas         401
Embraer                   151
Airbus                    138
Bombardier                 93
Boeing Stearman            50
Piper/Cub Crafters          7
Cessna Ector                6
Piper Aerostar              2
Cessna/Air Repair           2
Piper Cub Crafters          1
Mcdonnell-Douglas           1
Jetprop Dlx Piper           1
Piper/Wally'S Flyers        1
Bombardier Learjet          1
Name: count, dtype: int64

In [17]:
# KEEPING 'MAKES' WITH ENOUGH DATA (50+ ACCIDENTS)

# Count accidents per Make
make_counts = df['Make'].value_counts()

# Keep Makes with at least 50 accidents:
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)].copy()

print('Makes remaining:', df['Make'].nunique())
df.shape

Makes remaining: 65


(59808, 34)

**'Make' Cleaning Tasks:**
1. Standardized capitalization and spacing (e.g., "CESSNA" → "Cessna"), done earlier before imputing Aircraft.Category.
2. Dropped 1 row with a missing Make.
3. Removed company suffixes and punctuation (e.g., "Cessna Aircraft Co" → "Cessna", "Cirrus Design Corp." → "Cirrus"): 1,050 → 886 unique Makes.
4. Merged remaining name variations (e.g., "Hawker Beechcraft" → "Beech", "New Piper" → "Piper").
5. Intentionally kept "Boeing Stearman" and "Mcdonnell Douglas" separate from Boeing, since they are different aircraft lines.
6. Kept only Makes with 50+ accidents so comparisons are statistically reliable: **65 Makes, 59,808 rows** remain.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [18]:
# INSPECTING AND CLEANING 'MODEL'

# Drop rows with a missing Model:
print('Missing:', df['Model'].isna().sum())
df = df.dropna(subset=['Model']).copy()

# Standardize: remove extra spaces, make Model uppercase
df['Model'] = df['Model'].str.strip().str.upper()

# Most common models:
print(df['Model'].value_counts().head(10))

# Are the model names unique to each make? --> no
# Count the 'Make' per 'Model'
makes_per_model = df.groupby('Model')['Make'].nunique()
makes_per_model[makes_per_model > 1].sort_values(ascending=False).head(10)

Missing: 13
Model
152          2211
172          1614
172N         1080
PA-28-140     857
172M          755
150           742
172P          661
182           608
180           594
150M          550
Name: count, dtype: int64


Model
AA-5B     6
G-164A    6
500       6
690B      5
112       5
G-164B    5
G-164     5
AA-1B     5
AA-5      5
AA-5A     5
Name: Make, dtype: int64

In [19]:
# CREATING A UNIQUE PLANE IDENTIFIER:

# Model names repeat across different Makes, so combine Make + Model
df['Make.Model'] = df['Make'] + ' ' + df['Model']

print('Unique Make.Model:', df['Make.Model'].nunique())
df['Make.Model'].value_counts().head(10)


Unique Make.Model: 4809


Make.Model
Cessna 152         2211
Cessna 172         1614
Cessna 172N        1080
Piper PA-28-140     857
Cessna 172M         755
Cessna 150          742
Cessna 172P         661
Cessna 182          608
Cessna 180          593
Cessna 150M         550
Name: count, dtype: int64

**Model cleaning:**
- Dropped 13 rows with a missing Model.
- Standardized Model text (removed extra spaces, converted to uppercase).
- Model labels are **not** unique to each Make (e.g., "500" appears under 6 different Makes), so grouping by Model alone would combine unrelated aircraft.
- Created `Make.Model` (e.g., "Cessna 172N") as a unique identifier for each plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [20]:
# INSPECTING THE OTHER COLUMNS:

other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']

# Print the value counts (including NaN) for each column:
for col in other_cols:
    print(df[col].value_counts(dropna=False))
    print()

Engine.Type
Reciprocating      51807
NaN                 3042
Turbo Prop          2624
Turbo Fan           1337
Unknown              502
Turbo Jet            377
Turbo Shaft          104
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

Weather.Condition
VMC    52484
IMC     4800
NaN     1817
UNK      554
Unk      140
Name: count, dtype: int64

Number.of.Engines
1.0    48664
2.0     8104
NaN     2349
4.0      283
3.0      278
0.0      117
Name: count, dtype: int64

Purpose.of.flight
Personal                     35028
Instructional                 8056
Unknown                       3905
Aerial Application            3301
Business                      2958
NaN                           2810
Positioning                   1019
Ferry                          554
Other Work Use                 525
Aerial Observation             386
Executive/corporate            348
Public Aircraft                338
Skydiving                      171
Flight Test                   

In [21]:
# CLEANING THESE OTHER COLUMNS:

# Placeholders that mean missing/unknown --> NaN
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan})
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan})
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan})
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})

# An airplane with 0 engines aka unknown/missing --> NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)    # np.nan = numpy's "missing value"

# Merge duplicate or similar categories:
df['Engine.Type'] = df['Engine.Type'].replace({'Geared Turbofan': 'Turbo Fan'})
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race show': 'Air Race/Show',
    'Air Race/show': 'Air Race/Show',
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft'
})

# Group rare purposes (<50 accidents) together into 'Other'
purpose_counts = df['Purpose.of.flight'].value_counts()
rare_purposes = purpose_counts[purpose_counts < 50].index
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace(rare_purposes, 'Other')

# Checking results:
for col in other_cols:
    print(df[col].value_counts(dropna=False))
    print()


Engine.Type
Reciprocating    51807
NaN               3545
Turbo Prop        2624
Turbo Fan         1338
Turbo Jet          377
Turbo Shaft        104
Name: count, dtype: int64

Weather.Condition
VMC    52484
IMC     4800
NaN     2511
Name: count, dtype: int64

Number.of.Engines
1.0    48664
2.0     8104
NaN     2466
4.0      283
3.0      278
Name: count, dtype: int64

Purpose.of.flight
Personal               35028
Instructional           8056
NaN                     6715
Aerial Application      3301
Business                2958
Positioning             1019
Ferry                    554
Other Work Use           525
Public Aircraft          421
Aerial Observation       386
Executive/corporate      348
Skydiving                171
Flight Test              123
Banner Tow                97
Other                     93
Name: count, dtype: int64

Broad.phase.of.flight
NaN            16520
Landing        11983
Takeoff         8933
Cruise          7141
Maneuvering     4657
Approach        4581
T

**Cleaning other columns:**
- Converted placeholder values ("Unknown", "UNK", "Unk") to NaN in Engine.Type, Weather.Condition, Purpose.of.flight, and Broad.phase.of.flight, so they aren't treated as real categories.
- Number.of.Engines: 0 engines is not possible for an airplane, so 117 values were set to NaN.
- Merged duplicate/similar categories (e.g., "Geared Turbofan" → "Turbo Fan", "Public Aircraft - Federal/State/Local" → "Public Aircraft").
- Grouped Purpose.of.flight categories with fewer than 50 accidents into "Other".
- NaNs were kept (not dropped) so rows remain available for analyses that don't use these columns.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [22]:
# REMOVING COLUMNS WIETH TOO MANY NaN'

# Non-null count for each column (lowest first)
print(df.notna().sum().sort_values())

# Keep the columns with more than 20,000 non-null values:
keep_cols = df.columns[df.notna().sum() > 20000]

# Show which columns will be dropped:
print('\nDropped:', list(df.columns.difference(keep_cols)))

df = df[keep_cols].copy()
df.shape

Schedule                   7253
Air.carrier               10063
FAR.Description           17436
Longitude                 22193
Latitude                  22199
Airport.Code              36518
Airport.Name              38210
Broad.phase.of.flight     43275
Publication.Date          49430
Purpose.of.flight         53080
Engine.Type               56250
Report.Status             56678
Weather.Condition         57284
Number.of.Engines         57329
Registration.Number       59136
Country                   59703
Location                  59767
Model                     59795
Aircraft.Category         59795
Aircraft.damage           59795
Injury.Severity           59795
Accident.Number           59795
Investigation.Type        59795
Event.Id                  59795
Event.Date                59795
Make                      59795
Total.Fatal.Injuries      59795
Amateur.Built             59795
Total.Uninjured           59795
Total.Minor.Injuries      59795
Total.Serious.Injuries    59795
Total.Pa

(59795, 32)

**Column removal:** Dropped columns with 20,000 or fewer non-null values (Schedule, Air.carrier, FAR.Description), since they are too incomplete to support reliable analysis. Final cleaned dataset: **59,795 rows × 32 columns**.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [23]:
# SAVING THE CLEANED DATA

# Save to the data folder (index=False leaves out the row numbers)
df.to_csv('data/AviationData_Cleaned.csv', index=False)

# Check: reload the file and confirm its size matches
check = pd.read_csv('data/AviationData_Cleaned.csv', low_memory=False)
check.shape

(59795, 32)